# FLEX — every California park, all precip, all scenarios

Small flex of the library + Coiled stack: grab monthly precipitation for **every NPS unit in (or brushing) California**, across historical + all three SSPs, 1950–2100, all 15 LOCA2 models. Parallel across parks. One `get_climate_data` call per park, dispatched concurrently.

## A note on map-reduce, Rust, gRPC

Short answer: this is already map-reduce, just wearing Python clothes.

- `build_coiled_task` (in the library) creates a `dask.delayed` task per (park × scenario). That's the **map** step.
- `dask.compute(*tasks)` sends the whole task graph to the scheduler. Dask's scheduler is written in C-extensions + careful Python, and talks to workers over a tuned TCP protocol. Results stream back as they finish — that's the **reduce** step.
- Each worker runs arbitrary Python (xarray, numpy, pandas) with direct S3 access in the same region as the data. The bottleneck is S3 read throughput, not orchestration.

Dropping in a Rust process or a gRPC layer wouldn't help here — the scheduler/worker protocol is already the fastest part, and the per-task work is pandas/xarray ops you can't easily replace with Rust without rewriting the whole pipeline. Where Rust *would* help is per-cell numeric hot loops (e.g., a custom unit-conversion kernel), but we don't have one — Cal-Adapt's Zarr + xarray arithmetic is already vectorized C.

In [1]:
import os, sys, time
from time import perf_counter
import coiled
import pandas as pd
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed
from shapely.geometry import box

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "lib"))

from andrewAdaptLibrary import ParkCatalog, CatalogExplorer, get_climate_data


# ---- timing helper ----
class Timer:
    """Context manager: prints start/end and stashes .elapsed for later use."""
    def __init__(self, label): self.label = label
    def __enter__(self):
        self.start = perf_counter()
        print(f"▶ {self.label}…")
        return self
    def __exit__(self, *args):
        self.elapsed = perf_counter() - self.start
        print(f"  ✓ {self.label}: {self.elapsed:.1f}s ({self.elapsed/60:.2f} min)")


T_START = perf_counter()   # notebook-wide wall-clock anchor

NPS_SHP = os.path.join(
    PROJECT_ROOT,
    "USA_National_Park_Service_Lands_20170930_4993375350946852027",
    "USA_Federal_Lands.shp",
)
park_catalog = ParkCatalog(NPS_SHP)
cat = CatalogExplorer(timescale="monthly")       # one shared catalog for all fetches
print(park_catalog, "|", cat)

/opt/conda/envs/py-env/lib/python3.12/site-packages/pyogrio/raw.py:200: RuntimeWarning: /workspaces/DSEBrandNew/USA_National_Park_Service_Lands_20170930_4993375350946852027/USA_Federal_Lands.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


ParkCatalog(437 NPS units) | CatalogExplorer(activity='LOCA2', timescale='monthly', grid='d03', stores=1585)


## 1. Find every CA park — geometry on the mega NPS layer

The NPS shapefile has no state column, so we use geometry instead: any unit whose boundary intersects a California bounding box. Catches some edge-of-state units too (Great Basin, Lake Mead, Tule Springs) — fine, they're all in the LOCA2 grid.

In [2]:
CA_BBOX = box(-124.5, 32.5, -114.0, 42.0)
gdf = park_catalog._gdf                           # internal GDF; TODO lib helper
ca_names = sorted(gdf.loc[gdf.geometry.intersects(CA_BBOX), "unit_name"].unique())
print(f"{len(ca_names)} NPS units intersect California")
for n in ca_names: print(f"  {n}")

31 NPS units intersect California
  Cabrillo National Monument
  Castle Mountains National Monument
  Channel Islands National Park
  César E. Chávez National Monument
  Death Valley National Park
  Devils Postpile National Monument
  Eugene O'Neill National Historic Site
  Fort Point National Historic Site
  Golden Gate National Recreation Area
  Great Basin National Park
  John Muir National Historic Site
  Joshua Tree National Park
  Kings Canyon National Park
  Lake Mead National Recreation Area
  Lassen Volcanic National Park
  Lava Beds National Monument
  Manzanar National Historic Site
  Mojave National Preserve
  Muir Woods National Monument
  Pinnacles National Park
  Point Reyes National Seashore
  Port Chicago Naval Magazine National Memorial
  Redwood National Park
  Rosie the Riveter/World War II Home Front National Historical Park
  San Francisco Maritime National Historical Park
  Santa Monica Mountains National Recreation Area
  Sequoia National Park
  Tule Lake Nation

## 2. Safety probe — keep only parks fully inside the LOCA2 3km grid

`_inside_loca2(boundary)` is a fast geometric check (no network). Any park that fails it would need the WRF dynamical-downscaling grid instead, which the notebook doesn't handle here.

In [3]:
boundaries = {n: park_catalog.get_boundary(n) for n in ca_names}
ready   = [n for n, b in boundaries.items() if park_catalog._inside_loca2(b)]
skipped = [n for n in ca_names if n not in ready]
print(f"ready: {len(ready)}   skipped: {len(skipped)}")
if skipped: print("outside LOCA2:", skipped)

ready: 31   skipped: 0


## 3. Start the cluster — 45 workers

Sized for the task: 45 workers ÷ 4 scenarios per park ≈ 11 parks fetching in parallel at steady state. With ~31 parks total this keeps everyone busy.

In [4]:
with Timer("cluster spin-up") as cluster_timer:
    cluster = coiled.Cluster(
        name="flex-ca-precip",
        region="us-central1",
        n_workers=45,
        worker_vm_types=["n2-highmem-4"],
        spot_policy="spot_with_fallback",
        idle_timeout="30 minutes",
        package_sync=True,
    )
    client = cluster.get_client()

n_workers_actual = len(client.scheduler_info()['workers'])
print(f"  Workers ready: {n_workers_actual} / 45 requested")
print(f"  Dashboard: {client.dashboard_link}")

▶ cluster spin-up…


/tmp/ipykernel_58118/4195247294.py:2: FutureWarning: `package_sync` is a deprecated kwarg for `Cluster` and will be removed in a future release. To only sync certain packages, use `package_sync_only`, and to disable package sync, pass the `container` or `software` kwargs instead.
  cluster = coiled.Cluster(


Output()

╭──────────────────────────────── Package Info ────────────────────────────────╮
│                    ╷                                                         │
│   Package          │ Note                                                    │
│ ╶──────────────────┼───────────────────────────────────────────────────────╴ │
│   coiled_local_lib │ Source wheel built from /workspaces/DSEBrandNew/lib     │
│   climakitae       │ Wheel built from                                        │
│                    │ https://github.com/cal-adapt/climakitae/archive/refs/   │
│                    │ tags/1.4.0.zip                                          │
│                    ╵                                                         │
╰──────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────── Not Synced with Cluster ───────────────────────────╮
│             ╷                                                    ╷           │
│   Package   │ Error                                              │ Level     │
│ ╶───────────┼────────────────────────────────────────────────────┼─────────╴ │
│   shiboken6 │ cannot find shiboken6~=6.10.2 on pypi.org. If you  │ Warning   │
│             │ are using a custom PyPI URL, please see            │           │
│             │ https://docs.coiled.io/user_guide/software/packag… │           │
│             │ for more instructions.                             │           │
│             ╵                                                    ╵           │
╰──────────────────────────────────────────────────────────────────────────────╯

Output()

  ✓ cluster spin-up: 231.9s (3.87 min)
  Workers ready: 5 / 45 requested
  Dashboard: https://cluster-emebb.dask.host/_IVA2MXDzt7JM7zp/status


## 4. Map-reduce across all parks

Each `get_climate_data(...)` call dispatches 4 scenario-tasks to Dask and blocks on the result. Run those calls from a ThreadPoolExecutor and the scheduler sees `4 × len(ready)` tasks at once, free to schedule across all 45 workers. Threads here are just submission handles — the real parallelism happens in the Dask scheduler.

In [ ]:
scenarios = ["Historical Climate", "SSP 2-4.5", "SSP 3-7.0", "SSP 5-8.5"]

def fetch(name):
    t0 = perf_counter()
    df = get_climate_data(
        variables=["Precip"], scenarios=scenarios,
        boundary=boundaries[name], time_slice=(1950, 2100),
        timescale="monthly", backend="coiled",
        coiled_cluster=cluster, catalog=cat,
    )["Precip"].assign(park=name)
    return name, df, perf_counter() - t0

results = {}
per_park_times = {}

with Timer("all-park fetch (wall-clock)") as fetch_timer:
    with ThreadPoolExecutor(max_workers=len(ready)) as ex:
        futures = [ex.submit(fetch, name) for name in ready]
        for i, f in enumerate(as_completed(futures), 1):
            name, df, dt = f.result()
            results[name] = df
            per_park_times[name] = dt
            print(f"  [{i:2d}/{len(ready)}] {name[:45]:45s}  {dt:5.1f}s  {len(df):>7,} rows")

▶ all-park fetch (wall-clock)…


## 5. Reduce — concat everything into one tidy frame

In [ ]:
with Timer("concat + type coerce") as concat_timer:
    all_precip = pd.concat(results.values(), ignore_index=True)
    all_precip["time"] = pd.to_datetime(all_precip["time"])

print(f"  rows:       {len(all_precip):,}")
print(f"  parks:      {all_precip['park'].nunique()}")
print(f"  years:      {all_precip['time'].dt.year.min()}–{all_precip['time'].dt.year.max()}")
print(f"  scenarios:  {sorted(all_precip['scenario'].unique())}")
print(f"  simulations: {all_precip['simulation'].nunique()}")
print(f"  memory:     {all_precip.memory_usage(deep=True).sum() / 1e6:.0f} MB")
all_precip.head()

## 6. Quick look — historical annual precipitation per park

One bar per park: multi-model-mean annual total precip over 1985–2014, ranked low-to-high. Instant sanity check (coastal parks wet, desert parks dry) and answers the "which parks in CA actually get rain?" question in one plot.

In [ ]:
hist = all_precip.query("scenario == 'Historical Climate'").copy()
hist["year"] = hist["time"].dt.year
hist = hist[hist["year"].between(1985, 2014)]

# Per-park mean annual total: sum within (model, year), then mean over models + years
annual = hist.groupby(["park", "simulation", "year"])["Precip"].sum().reset_index()
park_mean = annual.groupby("park")["Precip"].mean().sort_values()

fig, ax = plt.subplots(figsize=(10, max(4, 0.3 * len(park_mean))))
park_mean.plot.barh(ax=ax, color="#4a90c2")
ax.set_xlabel("Mean annual total precipitation (mm/yr), 1985–2014")
ax.set_ylabel("")
ax.set_title("California NPS units, ranked by historical annual precipitation")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()

## 7. Cost + data-volume summary

How much did the cloud actually chomp through, and how long would the same thing have taken locally? The data-volume estimate leans on what we know about LOCA2 Zarr layout (time-chunked with full spatial extent per chunk, so any time-subset fetch effectively pulls whole stores) and the timing you just watched print out above.

In [ ]:
T_TOTAL = perf_counter() - T_START

# -- Data volume (LOCA2 chunk-layout-aware estimate) --
# LOCA2 monthly 3km stores are typically chunked by time with the full western-US
# grid per chunk. A time-subset fetch therefore pulls roughly the entire store.
N_MODELS     = 15
N_VARS       = 1
N_SCENARIOS  = 4
N_UNIQUE_STORES = N_MODELS * N_VARS * N_SCENARIOS
AVG_STORE_GB    = 1.26   # ~1.0 GB historical + ~1.33 GB per SSP, averaged
store_reads     = N_UNIQUE_STORES * len(results)        # one full read per (store × park)
data_chomped_gb = store_reads * AVG_STORE_GB

# -- Fetch timing stats --
t_slowest = max(per_park_times.values())
t_fastest = min(per_park_times.values())
t_avg     = sum(per_park_times.values()) / len(per_park_times)
t_serial  = sum(per_park_times.values())        # hypothetical sequential total

# -- Local-equivalent estimates --
# Mac on home wifi: ~50 MB/s sustained; network IS the bottleneck
MAC_MBPS = 50
mac_hours = (data_chomped_gb * 1000) / (MAC_MBPS * 3600)
# Linux workstation, fiber + async: ~200 MB/s effective, still single node
LINUX_MBPS = 200
linux_hours = (data_chomped_gb * 1000) / (LINUX_MBPS * 3600)

print("=" * 58)
print("  CLOUD COST / THROUGHPUT")
print("=" * 58)
print(f"  Parks processed:             {len(results)}")
print(f"  Workers actually up:         {n_workers_actual} / 45")
print(f"  Unique Zarr stores:          {N_UNIQUE_STORES}")
print(f"  Store-read operations:       {store_reads:,}")
print(f"  Data chomped through:        ~{data_chomped_gb:,.0f} GB  (~{data_chomped_gb/1000:.2f} TB)")
print(f"  Rows returned to notebook:   {len(all_precip):,}")
print(f"  In-memory footprint:         {all_precip.memory_usage(deep=True).sum() / 1e6:.0f} MB")
print()
print("=" * 58)
print("  TIMING")
print("=" * 58)
print(f"  Cluster spin-up:             {cluster_timer.elapsed:6.1f}s  ({cluster_timer.elapsed/60:4.1f} min)")
print(f"  Fetch wall-clock (parallel): {fetch_timer.elapsed:6.1f}s  ({fetch_timer.elapsed/60:4.1f} min)")
print(f"    ↳ avg per-park fetch:      {t_avg:6.1f}s")
print(f"    ↳ slowest park:            {t_slowest:6.1f}s  ({max(per_park_times, key=per_park_times.get)})")
print(f"    ↳ fastest park:            {t_fastest:6.1f}s  ({min(per_park_times, key=per_park_times.get)})")
print(f"    ↳ sum if run sequentially: {t_serial:6.1f}s  ({t_serial/60:4.1f} min) — parallelism saved {t_serial-fetch_timer.elapsed:.0f}s")
print(f"  Concat + coerce:             {concat_timer.elapsed:6.1f}s")
print(f"  Total wall-clock:            {T_TOTAL:6.1f}s  ({T_TOTAL/60:4.1f} min)")
print()
print("=" * 58)
print("  LOCAL-EQUIVALENT (download bottleneck only)")
print("=" * 58)
print(f"  MacBook on home wifi ({MAC_MBPS} MB/s):       ~{mac_hours:4.1f} hours  ({mac_hours/24:.1f} days)")
print(f"  Linux + fiber + async ({LINUX_MBPS} MB/s):    ~{linux_hours:4.1f} hours")
print(f"  — both would then add single-threaded xarray processing time on top.")
print()
print(f"  Cloud speedup vs MacBook:     ~{(mac_hours*3600) / T_TOTAL:.0f}×")

## 7. Shut down

In [ ]:
cluster.close()
print("Done.")